# Warehouse Signal — Pipeline Walkthrough

This notebook walks through every stage of the backend pipeline:

1. **Setup & Configuration** — env vars, DB state
2. **Universe** — which companies we track
3. **Transcript Providers** — mock vs FMP (live API)
4. **Parsing** — section detection (prepared remarks vs Q&A)
5. **Chunking** — breaking text into LLM-sized pieces
6. **Signal Extraction** — mock keyword vs Claude LLM
7. **Full Pipeline** — ingestion + analysis end-to-end
8. **Scoring** — composite score formula breakdown
9. **API Layer** — what the frontend sees
10. **Summary** — mock vs real comparison

---
## 1. Setup & Configuration

In [1]:
import sys
print(sys.executable)
print(sys.prefix)

/Users/devenyahraus/Documents/cre-stoatan/warehouse-signal/.venv/bin/python
/Users/devenyahraus/Documents/cre-stoatan/warehouse-signal/.venv


In [2]:
from warehouse_signal.storage.sqlite import Storage

storage = Storage()
stats = storage.get_stats()
print("=== Database State ===")
for k, v in stats.items():
    print(f"  {k:30s} : {v}")

=== Database State ===
  companies                      : 0
  transcripts                    : 0
  transcripts_unprocessed        : 0
  chunks                         : 0
  signal_extractions             : 0
  company_scores                 : 0


---
## 2. Universe — What Companies We Track

With an FMP API key, we fetch the full S&P 500 constituents. Without one, we fall back to a curated 30-ticker watchlist focused on warehouse-heavy companies.

In [3]:
from collections import Counter
from warehouse_signal.universe.sp500 import get_universe

companies = await get_universe()
print(f"Universe size: {len(companies)} companies\n")

# Sector distribution
sector_counts = Counter(c.sector.value for c in companies)
print("Sector distribution:")
for sector, count in sector_counts.most_common():
    print(f"  {sector:25s} : {count}")

# Show a few examples
print("\nSample companies:")
for c in companies[:10]:
    print(f"  {c.ticker:6s} {c.name:35s} [{c.sector.value}]")

Universe size: 30 companies

Sector distribution:
  retail                    : 7
  reit_industrial           : 5
  logistics_3pl             : 5
  ecommerce                 : 3
  food_distribution         : 3
  manufacturing             : 3
  reit_diversified          : 2
  automotive                : 2

Sample companies:
  PLD    Prologis Inc                        [reit_industrial]
  STAG   STAG Industrial Inc                 [reit_industrial]
  REXR   Rexford Industrial Realty           [reit_industrial]
  EGP    EastGroup Properties                [reit_industrial]
  FR     First Industrial Realty Trust       [reit_industrial]
  DLR    Digital Realty Trust                [reit_diversified]
  PSA    Public Storage                      [reit_diversified]
  XPO    XPO Inc                             [logistics_3pl]
  CHRW   C.H. Robinson Worldwide             [logistics_3pl]
  EXPD   Expeditors International            [logistics_3pl]


---
## 3. Transcript Providers — Where Data Comes From

Two providers:
- **MockProvider** — returns synthetic transcripts with embedded warehouse keywords (high/moderate/low signal templates)
- **FMPProvider** — fetches real earnings transcripts via the Financial Modeling Prep stable API

In [4]:
from warehouse_signal.providers.mock import MockProvider

mock_provider = MockProvider()
mock_transcript = await mock_provider.get_transcript("PLD", 2024, 3)

print("=== Mock Provider ===")
print(f"Provider: {mock_provider.name}")
print(f"Ticker: {mock_transcript.metadata.ticker}")
print(f"Quarter: {mock_transcript.metadata.year}Q{mock_transcript.metadata.quarter}")
print(f"Sections: {len(mock_transcript.sections)} ({mock_transcript.sections[0].section_type.value})")
print(f"Content length: {len(mock_transcript.raw_text)} chars")
print(f"\nFull mock transcript (HIGH_SIGNAL template):\n")
print(mock_transcript.raw_text)

=== Mock Provider ===
Provider: mock
Ticker: PLD
Quarter: 2024Q3
Sections: 1 (prepared_remarks)
Content length: 1424 chars

Full mock transcript (HIGH_SIGNAL template):

Thank you, operator. Good morning, everyone. I want to start by discussing our
distribution network strategy. As we communicated last quarter, we are actively
expanding our logistics footprint. We broke ground on two new distribution centers
this quarter — one in the Indianapolis market and one in the Inland Empire.

Our supply chain team has identified capacity constraints in our Midwest hub network.
Current utilization across our DCs is running at 94%, which is above our target of 85%.
To address this, we've committed approximately $180 million in logistics capex for the
next 18 months, focused on adding 2.4 million square feet of new warehouse capacity.

We're also seeing strong demand from our e-commerce channel. Direct-to-consumer fulfillment
now represents 38% of our total shipments, up from 29% a year ago. This 

In [5]:
from warehouse_signal.providers.fmp import FMPProvider

fmp_provider = FMPProvider()

# Fetch a real transcript
real_transcript = await fmp_provider.get_transcript("PLD", 2024, 3)

print("=== FMP Provider (Live API) ===")
print(f"Provider: {fmp_provider.name}")
print(f"Ticker: {real_transcript.metadata.ticker}")
print(f"Quarter: {real_transcript.metadata.year}Q{real_transcript.metadata.quarter}")
print(f"Call date: {real_transcript.metadata.call_date}")
print(f"Content length: {len(real_transcript.raw_text):,} chars")
print(f"Sections: {len(real_transcript.sections)} ({real_transcript.sections[0].section_type.value})")
print(f"\nFirst 500 chars:\n")
print(real_transcript.raw_text[:500])

=== FMP Provider (Live API) ===
Provider: fmp
Ticker: PLD
Quarter: 2024Q3
Call date: 2024-10-16
Content length: 50,281 chars
Sections: 1 (full)

First 500 chars:

Operator: Greetings, and welcome to the Prologis Q3 2024 Earnings Conference Call. At this time, all participants are in a listen-only mode. A question-and-answer session will follow the formal presentation. [Operator Instructions] And as a reminder, this conference is being recorded. It is now my pleasure to introduce you to your host, Justin Meng, SVP, Head of Investor Relations. Thank you. Justin, you may begin.
Justin Meng: Thanks, John, and good morning, everyone. Welcome to our third quart


In [6]:
# List available transcripts (probes each quarter — limited to 2024 for speed)
available = await fmp_provider.list_available_transcripts("PLD", start_year=2023, end_year=2024)

print(f"Available PLD transcripts (2023-2024): {len(available)}")
for m in available:
    print(f"  {m.year} Q{m.quarter}  (call date: {m.call_date})")

Available PLD transcripts (2023-2024): 8
  2024 Q4  (call date: 2025-01-21)
  2024 Q3  (call date: 2024-10-16)
  2024 Q2  (call date: 2024-07-17)
  2024 Q1  (call date: 2024-04-17)
  2023 Q4  (call date: 2024-01-17)
  2023 Q3  (call date: 2023-10-17)
  2023 Q2  (call date: 2023-07-18)
  2023 Q1  (call date: 2023-04-18)


---
## 4. Parsing — Section Detection

The parser looks for Q&A boundary patterns (e.g., "operator, open the line for questions") and splits the transcript into **prepared remarks** and **Q&A** sections.

FMP returns unsegmented content, so parsing adds structure.

In [7]:
from warehouse_signal.ingestion.parser import parse_sections

# Parse the real PLD transcript
parsed = parse_sections(real_transcript)

print(f"Sections found: {len(parsed.sections)}")
print(f"Has structured sections: {parsed.has_sections}\n")

for i, section in enumerate(parsed.sections):
    print(f"--- Section {i+1}: {section.section_type.value} ---")
    print(f"Length: {len(section.text):,} chars")
    print(f"First 200 chars: {section.text[:200]}...")
    print()

Sections found: 1
Has structured sections: False

--- Section 1: full ---
Length: 50,281 chars
First 200 chars: Operator: Greetings, and welcome to the Prologis Q3 2024 Earnings Conference Call. At this time, all participants are in a listen-only mode. A question-and-answer session will follow the formal presen...



---
## 5. Chunking — Breaking Text Into Analysis Units

Each section is split into chunks targeting ~800 tokens (configurable). Chunks:
- Never cross section boundaries
- Split on paragraph boundaries first, sentence boundaries as fallback
- Have deterministic IDs via SHA256(transcript_key::chunk_index)

In [8]:
from warehouse_signal.config import Config
from warehouse_signal.ingestion.parser import chunk_transcript

chunks = chunk_transcript(real_transcript)

print(f"Total chunks: {len(chunks)}")
print(f"Token distribution:")
tokens = [c.token_estimate for c in chunks]
print(f"  Min: {min(tokens)}, Max: {max(tokens)}, Avg: {sum(tokens)//len(tokens)}")
print(f"  Target: {Config.CHUNK_TARGET_TOKENS}, Max allowed: {Config.CHUNK_MAX_TOKENS}")
print()

# Show a few chunks
for chunk in chunks[:3]:
    print(f"--- Chunk {chunk.chunk_index} (id: {chunk.chunk_id}) ---")
    print(f"Section: {chunk.section_type.value}")
    print(f"Tokens: ~{chunk.token_estimate}")
    print(f"Text preview: {chunk.text[:150]}...")
    print()

Total chunks: 15
Token distribution:
  Min: 113, Max: 829, Avg: 762
  Target: 800, Max allowed: 1200

--- Chunk 0 (id: 3f5f0475a5fe07aa) ---
Section: full
Tokens: ~795
Text preview: Operator: Greetings, and welcome to the Prologis Q3 2024 Earnings Conference Call. At this time, all participants are in a listen-only mode. A questio...

--- Chunk 1 (id: 35a3f073141bce7d) ---
Section: full
Tokens: ~791
Text preview: We started over $0.5 billion in development projects, including incremental capital to an existing data center development now pre-leased to a hypersc...

--- Chunk 2 (id: 12fec9cb37990d1e) ---
Section: full
Tokens: ~783
Text preview: We remain very positive on the outlook for our business, as vacancies are still low in the context of history, starts are down significantly, and supp...



---
## 6. Signal Extraction — Mock vs Claude

Each chunk is analyzed to produce a `ChunkExtraction` with:
- `warehouse_relevance` (0-1): how relevant is this to warehouse/logistics?
- `expansion_score` (0-1): strength of expansion signal
- `move_type`: expansion / consolidation / relocation / optimization
- `time_horizon`: immediate / near_term / medium_term / long_term
- `signals`: boolean flags (capex, build-to-suit, last-mile, etc.)
- `geographic_mentions`: which regions are discussed
- `evidence_quote`: the key sentence supporting the scores

In [9]:
# Pick a chunk from the real transcript to analyze
# Use the first chunk which should be from prepared remarks
test_chunk = chunks[0]
print(f"Test chunk ({test_chunk.section_type.value}, ~{test_chunk.token_estimate} tokens):")
print(test_chunk.text[:300])
print("...")

Test chunk (full, ~795 tokens):
Operator: Greetings, and welcome to the Prologis Q3 2024 Earnings Conference Call. At this time, all participants are in a listen-only mode. A question-and-answer session will follow the formal presentation. [Operator Instructions] And as a reminder, this conference is being recorded. It is now my p
...


In [10]:
from warehouse_signal.analysis.mock import MockAnalyzer

mock_analyzer = MockAnalyzer()

mock_result = await mock_analyzer.extract_signals(test_chunk, "PLD", "Prologis Inc", 2024, 3)

print("=== Mock Analyzer (keyword matching) ===")
print(f"warehouse_relevance : {mock_result.warehouse_relevance}")
print(f"expansion_score     : {mock_result.expansion_score}")
print(f"move_type           : {mock_result.move_type.value}")
print(f"time_horizon        : {mock_result.time_horizon.value}")
print(f"reasoning           : {mock_result.reasoning}")
print(f"\nSignal flags:")
for k, v in mock_result.signals.model_dump().items():
    if v and v != 'none' and v != 'stable' and v != 'neutral':
        print(f"  {k}: {v}")
print(f"\nGeographic mentions: {[g.region for g in mock_result.geographic_mentions]}")
print(f"Evidence: {mock_result.evidence_quote[:150]}")

=== Mock Analyzer (keyword matching) ===
warehouse_relevance : 0.0
expansion_score     : 0.0
move_type           : unknown
time_horizon        : near_term
reasoning           : Keyword hits: 0/14

Signal flags:

Geographic mentions: []
Evidence: Operator: Greetings, and welcome to the Prologis Q3 2024 Earnings Conference Call.


In [11]:
from warehouse_signal.analysis.extractor import ClaudeAnalyzer

claude_analyzer = ClaudeAnalyzer()

claude_result = await claude_analyzer.extract_signals(test_chunk, "PLD", "Prologis Inc", 2024, 3)

print("=== Claude Analyzer (LLM extraction) ===")
print(f"warehouse_relevance : {claude_result.warehouse_relevance}")
print(f"expansion_score     : {claude_result.expansion_score}")
print(f"move_type           : {claude_result.move_type.value}")
print(f"time_horizon        : {claude_result.time_horizon.value}")
print(f"reasoning           : {claude_result.reasoning}")
print(f"\nSignal flags:")
for k, v in claude_result.signals.model_dump().items():
    if v and v != 'none' and v != 'stable' and v != 'neutral':
        print(f"  {k}: {v}")
print(f"\nGeographic mentions:")
for g in claude_result.geographic_mentions:
    print(f"  {g.region} (conf: {g.confidence}) — {g.context}")
print(f"\nEvidence: {claude_result.evidence_quote}")

=== Claude Analyzer (LLM extraction) ===
warehouse_relevance : 0.75
expansion_score     : 0.55
move_type           : optimization
time_horizon        : unspecified
reasoning           : This excerpt is highly relevant to warehouse/logistics real estate as it discusses Prologis' Q3 2024 portfolio performance, occupancy metrics, and lease mark-to-market dynamics specific to industrial properties. The expansion_score is moderate because while the text confirms active capital deployment and positive operational trends (96.2% occupancy, 34% mark-to-market), it lacks specific forward-looking warehouse expansion details, square footage commitments, or particular facility development plans.

Signal flags:
  capex_expansion: True
  vacancy_mention: True
  rent_pressure: upward
  construction_pipeline: active

Geographic mentions:

Evidence: In terms of deployment, we had a very active quarter.


In [12]:
# Side-by-side comparison
print(f"{'Field':<25} {'Mock':>15} {'Claude':>15}")
print("-" * 57)
print(f"{'warehouse_relevance':<25} {mock_result.warehouse_relevance:>15.2f} {claude_result.warehouse_relevance:>15.2f}")
print(f"{'expansion_score':<25} {mock_result.expansion_score:>15.2f} {claude_result.expansion_score:>15.2f}")
print(f"{'move_type':<25} {mock_result.move_type.value:>15} {claude_result.move_type.value:>15}")
print(f"{'time_horizon':<25} {mock_result.time_horizon.value:>15} {claude_result.time_horizon.value:>15}")
print(f"{'# geo mentions':<25} {len(mock_result.geographic_mentions):>15} {len(claude_result.geographic_mentions):>15}")
print(f"{'capex_expansion':<25} {str(mock_result.signals.capex_expansion):>15} {str(claude_result.signals.capex_expansion):>15}")
print(f"{'build_to_suit':<25} {str(mock_result.signals.build_to_suit):>15} {str(claude_result.signals.build_to_suit):>15}")
print(f"{'last_mile_expansion':<25} {str(mock_result.signals.last_mile_expansion):>15} {str(claude_result.signals.last_mile_expansion):>15}")

Field                                Mock          Claude
---------------------------------------------------------
warehouse_relevance                  0.00            0.75
expansion_score                      0.00            0.55
move_type                         unknown    optimization
time_horizon                    near_term     unspecified
# geo mentions                          0               0
capex_expansion                     False            True
build_to_suit                       False           False
last_mile_expansion                 False           False


In [13]:
# Show the actual prompt sent to Claude
from warehouse_signal.analysis.prompt import format_system_prompt, format_extraction_prompt

system = format_system_prompt("PLD", "Prologis Inc", 2024, 3, test_chunk.section_type.value)
user = format_extraction_prompt(test_chunk.text)

print("=== System Prompt ===")
print(system)
print("\n=== User Prompt (first 500 chars) ===")
print(user[:500])
print("...")

=== System Prompt ===
You are a commercial real estate analyst specializing in industrial/warehouse properties. Your job is to analyze earnings call transcript excerpts and identify signals related to warehouse, distribution center, and logistics facility expansion, consolidation, or relocation.

You are evaluating text from a full section of a PLD (Prologis Inc) earnings call for 2024Q3.

Respond ONLY with a JSON object matching the exact schema below. No markdown, no explanation outside the JSON.

=== User Prompt (first 500 chars) ===
Analyze this earnings call excerpt for warehouse and logistics real estate signals.

<transcript_chunk>
Operator: Greetings, and welcome to the Prologis Q3 2024 Earnings Conference Call. At this time, all participants are in a listen-only mode. A question-and-answer session will follow the formal presentation. [Operator Instructions] And as a reminder, this conference is being recorded. It is now my pleasure to introduce you to your host, Justin Meng, S

---
## 7. Full Ingestion + Analysis Pipeline

The complete pipeline for a single company-quarter:
1. `ingest_transcript()` — fetch, parse, chunk, store in DB
2. `analyze_transcript()` — extract signals from each chunk, store extractions

In [14]:
from warehouse_signal.ingestion.pipeline import ingest_transcript
from warehouse_signal.analysis.pipeline import analyze_transcript

# Use a fresh DB for this demo to avoid conflicts
demo_storage = Storage("data/notebook_demo.db")

print("=== Step 1: Ingest PLD 2024 Q3 ===")
ingested = await ingest_transcript(fmp_provider, demo_storage, "PLD", 2024, 3, force=True)
if ingested:
    print(f"Stored: {ingested.quarter_key}")
    print(f"Raw text: {len(ingested.raw_text):,} chars")
    print(f"Sections: {len(ingested.sections)}")
else:
    print("Already stored or failed")

# Check DB state after ingestion
print("\nDB after ingestion:")
for k, v in demo_storage.get_stats().items():
    print(f"  {k}: {v}")

=== Step 1: Ingest PLD 2024 Q3 ===
Stored: PLD_2024Q3
Raw text: 50,281 chars
Sections: 1

DB after ingestion:
  companies: 1
  transcripts: 1
  transcripts_unprocessed: 1
  chunks: 15
  signal_extractions: 15
  company_scores: 1


In [15]:
# Show chunks stored in DB
db_chunks = demo_storage.get_chunks_for_transcript("PLD_2024Q3")
print(f"Chunks in DB: {len(db_chunks)}")
for c in db_chunks[:3]:
    print(f"  chunk {c['chunk_index']}: {c['section_type']}, ~{c['token_estimate']} tokens")

Chunks in DB: 15
  chunk 0: full, ~795 tokens
  chunk 1: full, ~791 tokens
  chunk 2: full, ~783 tokens


In [16]:
print("=== Step 2: Analyze with Claude ===")
print(f"Analyzing {len(db_chunks)} chunks (this makes {len(db_chunks)} API calls)...\n")

# Get the transcript row for the analysis pipeline
transcript_row = list(demo_storage.db["transcripts"].rows_where(
    "quarter_key = ?", ["PLD_2024Q3"]
))[0]

num_extractions = await analyze_transcript(claude_analyzer, demo_storage, transcript_row)

print(f"Extractions saved: {num_extractions}")
print("\nDB after analysis:")
for k, v in demo_storage.get_stats().items():
    print(f"  {k}: {v}")

=== Step 2: Analyze with Claude ===
Analyzing 15 chunks (this makes 15 API calls)...

Extractions saved: 15

DB after analysis:
  companies: 1
  transcripts: 1
  transcripts_unprocessed: 0
  chunks: 15
  signal_extractions: 15
  company_scores: 1


In [17]:
import json

# Show raw extractions from DB
extractions = demo_storage.get_extractions_for_transcript("PLD_2024Q3")
print(f"Extractions for PLD 2024Q3: {len(extractions)}\n")

# Show the top 5 by expansion_score
sorted_ext = sorted(extractions, key=lambda e: e['expansion_score'], reverse=True)
for e in sorted_ext[:5]:
    llm = json.loads(e.get('raw_llm_output', '{}'))
    print(f"  chunk {e['chunk_id'][:8]}... | relevance={e['warehouse_relevance']:.2f} | expansion={e['expansion_score']:.2f} | {e['move_type']} | {e['time_horizon']}")
    print(f"    evidence: {llm.get('evidence_quote', '')[:100]}")
    print()

Extractions for PLD 2024Q3: 15

  chunk c95d7abe... | relevance=0.85 | expansion=0.72 | expansion | medium_term
    evidence: And then, let me finish with the fact that we have a very large development book right now, $5.5 bil

  chunk aeb638fb... | relevance=0.75 | expansion=0.70 | expansion | near_term
    evidence: We own land in about 50 -- over 50 markets globally. The average vintage year is about 4.5 years old

  chunk 83de3438... | relevance=0.75 | expansion=0.70 | expansion | near_term
    evidence: And so, yes, we are leasing with these customers, and yes, we think they will continue to lease spac

  chunk 35a3f073... | relevance=0.72 | expansion=0.65 | consolidation | medium_term
    evidence: Many customers are making progress in reducing this capacity through growth, while others are gainin

  chunk 12fec9cb... | relevance=0.75 | expansion=0.55 | expansion | medium_term
    evidence: We are reducing our overall development starts guidance to a range of $1.75 billion to $2.

---
## 8. Scoring — From Chunks to Company Score

The composite score formula:

```
composite = 0.40 * max_expansion_score
          + 0.30 * weighted_avg_expansion (weighted by warehouse_relevance)
          + 0.15 * signal_flag_bonus (0.05 each for capex, BTS, last-mile)
          + 0.15 * time_horizon_bonus (immediate=1.0, near=0.8, medium=0.5, long=0.3)
```

Only chunks with `warehouse_relevance >= 0.3` are considered "relevant".

In [18]:
from warehouse_signal.scoring.aggregator import score_company, compute_composite_score, RELEVANCE_THRESHOLD, TIME_WEIGHTS

# First, we need PLD in the companies table for sector lookup
from warehouse_signal.models.schemas import Company, Sector
demo_storage.upsert_company(Company(ticker="PLD", name="Prologis Inc", sector=Sector.REIT_INDUSTRIAL))

score = score_company(demo_storage, "PLD")

print("=== PLD Company Score ===")
print(f"Composite score     : {score.composite_score:.3f} ({score.composite_score*100:.0f}/100)")
print(f"Avg relevance       : {score.avg_warehouse_relevance:.3f}")
print(f"Avg expansion       : {score.avg_expansion_score:.3f}")
print(f"Max expansion       : {score.max_expansion_score:.3f}")
print(f"Relevant chunks     : {score.num_relevant_chunks} / {score.total_chunks}")
print(f"Move type           : {score.dominant_move_type.value}")
print(f"Time horizon        : {score.dominant_time_horizon.value}")
print(f"Top geographies     : {score.top_geographies}")
print(f"Signal flags        : capex={score.has_capex_signal}, BTS={score.has_build_to_suit}, last-mile={score.has_last_mile}")
print(f"\nEvidence snippets:")
for i, snippet in enumerate(score.evidence_snippets, 1):
    print(f"  {i}. {snippet[:120]}..." if len(snippet) > 120 else f"  {i}. {snippet}")

=== PLD Company Score ===
Composite score     : 0.556 (56/100)
Avg relevance       : 0.715
Avg expansion       : 0.534
Max expansion       : 0.720
Relevant chunks     : 14 / 15
Move type           : expansion
Time horizon        : medium_term
Top geographies     : ['Southern_California', 'Mexico', 'US_National', 'Inland_Empire', 'US']
Signal flags        : capex=True, BTS=True, last-mile=True

Evidence snippets:
  1. And then, let me finish with the fact that we have a very large development book right now, $5.5 billion underway on Pro...
  2. We own land in about 50 -- over 50 markets globally. The average vintage year is about 4.5 years old, 5 years old. And t...
  3. And so, yes, we are leasing with these customers, and yes, we think they will continue to lease space into next year and...


In [19]:
# Break down the formula with actual numbers
relevant = [e for e in extractions if e['warehouse_relevance'] >= RELEVANCE_THRESHOLD]

max_exp = max(e['expansion_score'] for e in relevant) if relevant else 0
total_weight = sum(e['warehouse_relevance'] for e in relevant)
weighted_avg = sum(e['expansion_score'] * e['warehouse_relevance'] for e in relevant) / total_weight if total_weight > 0 else 0

has_capex = any(json.loads(e.get('signals_json', '{}')).get('signals', {}).get('capex_expansion') for e in relevant)
has_bts = any(json.loads(e.get('signals_json', '{}')).get('signals', {}).get('build_to_suit') for e in relevant)
has_lm = any(json.loads(e.get('signals_json', '{}')).get('signals', {}).get('last_mile_expansion') for e in relevant)
flag_bonus = 0.05 * sum([has_capex, has_bts, has_lm])

time_scores = [TIME_WEIGHTS.get(e['time_horizon'], 0.2) for e in relevant]
time_bonus = sum(time_scores) / len(time_scores) if time_scores else 0

print("=== Score Formula Breakdown ===")
print(f"Relevant chunks: {len(relevant)} / {len(extractions)} (threshold: {RELEVANCE_THRESHOLD})")
print()
print(f"Component 1 (40%): max_expansion = {max_exp:.3f}")
print(f"  Contribution: 0.40 * {max_exp:.3f} = {0.40 * max_exp:.3f}")
print()
print(f"Component 2 (30%): weighted_avg_expansion = {weighted_avg:.3f}")
print(f"  Contribution: 0.30 * {weighted_avg:.3f} = {0.30 * weighted_avg:.3f}")
print()
print(f"Component 3 (15%): signal_flags = capex={has_capex}, BTS={has_bts}, last-mile={has_lm}")
print(f"  flag_bonus = 0.05 * {sum([has_capex, has_bts, has_lm])} = {flag_bonus:.3f}")
print(f"  Contribution: 0.15 * {flag_bonus:.3f} = {0.15 * flag_bonus:.4f}")
print()
print(f"Component 4 (15%): time_bonus = avg({[round(t,2) for t in time_scores[:5]]}...) = {time_bonus:.3f}")
print(f"  Contribution: 0.15 * {time_bonus:.3f} = {0.15 * time_bonus:.3f}")
print()
total = 0.40 * max_exp + 0.30 * weighted_avg + 0.15 * flag_bonus + 0.15 * time_bonus
print(f"TOTAL: {0.40*max_exp:.3f} + {0.30*weighted_avg:.3f} + {0.15*flag_bonus:.4f} + {0.15*time_bonus:.3f} = {total:.3f}")

=== Score Formula Breakdown ===
Relevant chunks: 14 / 15 (threshold: 0.3)

Component 1 (40%): max_expansion = 0.720
  Contribution: 0.40 * 0.720 = 0.288

Component 2 (30%): weighted_avg_expansion = 0.537
  Contribution: 0.30 * 0.537 = 0.161

Component 3 (15%): signal_flags = capex=True, BTS=True, last-mile=True
  flag_bonus = 0.05 * 3 = 0.150
  Contribution: 0.15 * 0.150 = 0.0225

Component 4 (15%): time_bonus = avg([0.2, 0.5, 0.5, 0.5, 0.5]...) = 0.564
  Contribution: 0.15 * 0.564 = 0.085

TOTAL: 0.288 + 0.161 + 0.0225 + 0.085 = 0.556


---
## 9. API Layer — What the Frontend Sees

The FastAPI server exposes these endpoints. Here we call the underlying functions directly.

In [20]:
# /api/stats
print("=== /api/stats ===")
print(json.dumps(demo_storage.get_stats(), indent=2))

=== /api/stats ===
{
  "companies": 1,
  "transcripts": 1,
  "transcripts_unprocessed": 0,
  "chunks": 15,
  "signal_extractions": 15,
  "company_scores": 1
}


In [21]:
# /api/scores/{ticker}
from warehouse_signal.scoring.aggregator import score_company, score_all_companies

pld_score = score_company(demo_storage, "PLD")
print("=== /api/scores/PLD ===")
print(json.dumps(pld_score.model_dump(mode='json'), indent=2, default=str))

=== /api/scores/PLD ===
{
  "ticker": "PLD",
  "company_name": "Prologis Inc",
  "sector": "reit_industrial",
  "composite_score": 0.556,
  "avg_warehouse_relevance": 0.715,
  "avg_expansion_score": 0.534,
  "max_expansion_score": 0.72,
  "num_relevant_chunks": 14,
  "total_chunks": 15,
  "top_geographies": [
    "Southern_California",
    "Mexico",
    "US_National",
    "Inland_Empire",
    "US"
  ],
  "dominant_time_horizon": "medium_term",
  "dominant_move_type": "expansion",
  "has_capex_signal": true,
  "has_build_to_suit": true,
  "has_last_mile": true,
  "evidence_snippets": [
    "And then, let me finish with the fact that we have a very large development book right now, $5.5 billion underway on Prologis share, and that's 33 million square feet of development underway. And then, we also have $40 billion worth of opportunities right now in our land bank.",
    "We own land in about 50 -- over 50 markets globally. The average vintage year is about 4.5 years old, 5 years old. And

In [22]:
# /api/scores/{ticker}/extractions — per-chunk detail
print("=== /api/scores/PLD/extractions (first 3) ===")
ext_rows = demo_storage.get_extractions_for_ticker("PLD")
for row in sorted(ext_rows, key=lambda r: r['expansion_score'], reverse=True)[:3]:
    print(json.dumps({
        'chunk_id': row['chunk_id'][:8] + '...',
        'warehouse_relevance': row['warehouse_relevance'],
        'expansion_score': row['expansion_score'],
        'move_type': row['move_type'],
        'time_horizon': row['time_horizon'],
    }, indent=2))
    print()

=== /api/scores/PLD/extractions (first 3) ===
{
  "chunk_id": "c95d7abe...",
  "warehouse_relevance": 0.85,
  "expansion_score": 0.72,
  "move_type": "expansion",
  "time_horizon": "medium_term"
}

{
  "chunk_id": "aeb638fb...",
  "warehouse_relevance": 0.75,
  "expansion_score": 0.7,
  "move_type": "expansion",
  "time_horizon": "near_term"
}

{
  "chunk_id": "83de3438...",
  "warehouse_relevance": 0.75,
  "expansion_score": 0.7,
  "move_type": "expansion",
  "time_horizon": "near_term"
}



In [23]:
# /api/geographies
# Save score first so it appears in the geographies view
demo_storage.save_company_score(pld_score)

all_scores_raw = demo_storage.get_all_company_scores()
all_scores = [demo_storage.row_to_company_score(r) for r in all_scores_raw]

geo_data = {}
for s in all_scores:
    for geo in s.top_geographies:
        geo_data.setdefault(geo, []).append(s.composite_score)

print("=== /api/geographies ===")
for region in sorted(geo_data, key=lambda g: sum(geo_data[g])/len(geo_data[g]), reverse=True):
    scores = geo_data[region]
    print(f"  {region:25s}  avg={sum(scores)/len(scores):.3f}  companies={len(scores)}")

=== /api/geographies ===
  Southern_California        avg=0.556  companies=1
  Mexico                     avg=0.556  companies=1
  US_National                avg=0.556  companies=1
  Inland_Empire              avg=0.556  companies=1
  US                         avg=0.556  companies=1


---
## 10. Summary — Mock vs Real

| Aspect | Mock Mode | Real Mode |
|--------|-----------|----------|
| Transcript source | 3 synthetic templates (~500 chars each) | FMP API (real earnings calls, 30-50K chars) |
| Section parsing | Mock returns pre-labeled sections | Parser detects Q&A boundary via regex |
| Chunking | 1-2 chunks per transcript | 30-60+ chunks per real transcript |
| Signal extraction | Keyword counting (deterministic) | Claude LLM analysis (nuanced, contextual) |
| Geographic detection | 5 hardcoded regex patterns | Claude identifies arbitrary regions from context |
| Scoring formula | Same formula, but shallow input | Same formula, deep/reliable input |
| API cost | $0 | ~$0.02-0.05 per transcript (Claude) + FMP subscription |

The pipeline architecture is identical in both modes — only the provider and analyzer are swapped.

In [24]:
# Clean up
await fmp_provider.close()
await claude_analyzer.close()
print("Done. Providers closed.")

Done. Providers closed.
